In [2]:
from typing import List
import pickle
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl


In [3]:
class Module:
	def forward(self, x):
		raise NotImplementedError

	def backward(self, grad):
		raise NotImplementedError

	def adjust(self, eta):
		raise NotImplementedError


class LinearLayer(Module):
	def __init__(self, input_size: int, output_size: int, weights: None | np.ndarray = None,
	             biases: None | np.ndarray = None, min_weight:None|float = None ,max_weight:None|float = None):
		if weights is not None:
			if weights.shape != (output_size, input_size):
				raise ValueError("niezgodne wymiary macierzy wag")
			self.W = np.copy(weights)
		else:
			self.W = np.random.uniform(min_weight, max_weight, (output_size, input_size))

		if biases is not None:
			biases = np.atleast_2d(biases)
			if biases.shape != (1, output_size):
					raise ValueError("niezgodne wymiary macierzy biasow")
			self.b = np.copy(biases)
		else:
			self.b = None
		self.cache_X = None
		self.cache_grad = None

	def forward(self, x):
		self.cache_X = np.atleast_2d(x)
		if self.b is not None:
			return self.cache_X @ self.W.T + self.b

		return self.cache_X @ self.W.T
	def backward(self, grad):
		self.cache_grad = grad
		return grad @ self.W

	def adjust(self, eta):
		grad_W = self.cache_grad.T @ self.cache_X
		self.W -= eta * grad_W
		if self.b is not None:
			grad_b = np.sum(self.cache_grad, axis=0, keepdims=True)
			self.b -= eta * grad_b


class ActivationLayer(Module):

	@staticmethod
	def relu(x: np.ndarray) -> np.ndarray:
		return np.maximum(x, 0)

	@staticmethod
	def relu_derivative(x: np.ndarray) -> np.ndarray:
		return np.where(x > 0, 1.0, 0)

	ACTIVATION_FUNCTIONS = {
		'ReLU': (relu, relu_derivative)
	}

	def __init__(self, activation_function):
		self.activation_function = self.ACTIVATION_FUNCTIONS[activation_function][0]
		self.derivative_function = self.ACTIVATION_FUNCTIONS[activation_function][1]
		self.cache_S = None

	def forward(self, S):
		self.cache_S = S
		return self.activation_function(S)

	def backward(self, grad):
		return grad * self.derivative_function(self.cache_S)

	def adjust(self, eta):
		pass


class NeuralNetworkException(Exception):
	def __init__(self, msg):
		self.messege = msg

	def get_message(self):
		return self.messege


class NeuralNetwork:

	def __init__(self, initial_input_size):
		self.layers: list[Module] = list()
		self.initial_input_size = initial_input_size
		self.last_layer_input_size = initial_input_size

	def add_layer(self, layer: Module):
		self.layers.append(layer)

	def save_model(self, filename: str):
		params = {}
		for i, layer in enumerate(self.layers):
			if isinstance(layer, LinearLayer):
				params[f'W_{i}'] = layer.W
				if layer.b is not None:
					params[f'b_{i}'] = layer.b
		np.savez(filename, **params)

	def load_model(self, filename: str):
		data = np.load(filename if filename.endswith('.npz') else filename + '.npz')
		for i, layer in enumerate(self.layers):
			if isinstance(layer, LinearLayer):
				layer.W = data[f'W_{i}']
				if f'b_{i}' in data:
					layer.b = data[f'b_{i}']

	def predict(self, x: np.ndarray | None):
		output = x
		for layer in self.layers:
			output = layer.forward(output)

		return output

	def fit(self, x, y, learning_rate) -> float:
		prediction = self.predict(x)
		error = prediction - y
		loss = float(np.mean(np.square(error)))
		grad = (2.0 / error.size) * error
		for layer in reversed(self.layers):
			grad = layer.backward(grad)

		for layer in self.layers:
			layer.adjust(learning_rate)

		return loss

	def evaluate(self,x,y):
		data_size = y.shape[0]
		correct_predictions_count = 0
		false = []
		for i in range(data_size):
			predicted_num = np.argmax(self.predict(x[i]))
			expected_num = y[i]
			if int(predicted_num) == int(expected_num):
				correct_predictions_count += 1
			else:
				false.append(i)
		return ((correct_predictions_count / data_size), false)




In [4]:
x_input = np.array([[0.5, 0.1, 0.2, 0.8],
					[0.75, 0.3, 0.1, 0.9],
					[0.1, 0.7, 0.6, 0.2]])

y_expected = np.array([[0.1, 0.5, 0.1, 0.7],
					   [1.0, 0.2, 0.3, 0.6],
					   [0.1, -0.5, 0.2, 0.2]
					   ])

Wh = np.array([
	[0.1, 0.1, -0.3],
	[0.1, 0.2, 0.0],
	[0.0, 0.7, 0.1],
	[0.2, 0.4, 0.0],
	[-0.3, 0.5, 0.1]
])
Wy = np.array([
	[0.7, 0.9, -0.4, 0.8, 0.1],
	[0.8, 0.5, 0.3, 0.1, 0.0],
	[-0.3, 0.9, 0.3, 0.1, -0.2]
])

alpha = 0.01

ReLu_model = NeuralNetwork(initial_input_size=3)

ReLu_model.add_layer(LinearLayer(3, 5,Wh))
ReLu_model.add_layer(ActivationLayer("ReLU"))
ReLu_model.add_layer(LinearLayer(5, 3,Wy))

epoch_count = 50
stop_epochs = [0, 49]
for epoch_number in range(epoch_count):
	if epoch_number in stop_epochs:
		print(f"Oczekiwane wynik w {epoch_number + 1} epoce:\n")

	total_loss = 0
	for i in range(x_input.shape[1]):
		series_x = x_input[:, i]

		out = ReLu_model.predict(series_x)
		epoch_loss = ReLu_model.fit(x=series_x, y=y_expected[:, i], learning_rate=alpha)
		if epoch_number in stop_epochs:
			print(f" * wyjście sieci po {i + 1} serii - output = \n{out.reshape(-1, 1)}\n")
			total_loss += epoch_loss

	if epoch_number in stop_epochs:
		print(f"totalny loss: {total_loss}")



Oczekiwane wynik w 1 epoce:

 * wyjście sieci po 1 serii - output = 
[[0.376 ]
 [0.3765]
 [0.305 ]]

 * wyjście sieci po 2 serii - output = 
[[0.08071923]
 [0.13408188]
 [0.12250318]]

 * wyjście sieci po 3 serii - output = 
[[0.05315396]
 [0.06705642]
 [0.07174004]]

 * wyjście sieci po 4 serii - output = 
[[0.49049287]
 [0.47019548]
 [0.39857572]]

totalny loss: 0.415895892053679
Oczekiwane wynik w 50 epoce:

 * wyjście sieci po 1 serii - output = 
[[0.44275608]
 [0.56549362]
 [0.15730947]]

 * wyjście sieci po 2 serii - output = 
[[0.14041801]
 [0.18610531]
 [0.06389472]]

 * wyjście sieci po 3 serii - output = 
[[0.10559427]
 [0.09365885]
 [0.04504851]]

 * wyjście sieci po 4 serii - output = 
[[0.58453677]
 [0.71142718]
 [0.2163731 ]]

totalny loss: 0.28322173534413086


In [5]:

def open_ghost_data(file_name: str):
	inputs = []
	expected_outputs = []
	with open(file_name, "r", encoding="utf-8") as file:
		for line in file:
			parsed_line = line.split()

			if len(parsed_line) >= 4:
				inputs.append([float(x) for x in parsed_line[0:3]])
				expected_outputs.append(int(parsed_line[3]) - 1)
	return np.array(inputs), np.array(expected_outputs)


train_inputs, train_outputs = open_ghost_data("../data/colors_train.txt")
test_inputs, test_outputs = open_ghost_data("../data/colors_test.txt")



color_model = NeuralNetwork(initial_input_size=3)
color_model.add_layer(LinearLayer(3, 8, min_weight=0.01,max_weight=0.99))
color_model.add_layer(ActivationLayer("ReLU"))
color_model.add_layer(LinearLayer(8, 4,min_weight=0.01,max_weight=0.99))

epoch_count = 200
alpha = 0.01
for epoch_number in range(epoch_count):
	epoch_error = 0
	for n in range(train_outputs.shape[0]):
		y_temp = np.zeros(4)
		y_temp[int(train_outputs[n])] = 1
		epoch_error += color_model.fit(train_inputs[n], y_temp, learning_rate=alpha)


correctness , _ = color_model.evaluate(test_inputs, test_outputs)
print(f"popraność: {correctness}")



popraność: 1.0


In [6]:
import pandas
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', version=1, as_frame=False)
x_data, y_data = mnist.data, mnist.target
x_data = x_data / 255

In [7]:

x_train, x_test = x_data[:10000], x_data[10000:11000]
y_train, y_test = y_data[:10000], y_data[10000:11000]
print(f"x_train: {x_train.shape}, y_train: {y_train.shape}")
print(f"x_test: {x_test.shape}, y_test: {y_test.shape}")


x_train: (10000, 784), y_train: (10000,)
x_test: (1000, 784), y_test: (1000,)


In [8]:

mnist_model = NeuralNetwork(784)
mnist_model.add_layer(LinearLayer(784, 40,min_weight=-0.1,max_weight=0.1))
mnist_model.add_layer(ActivationLayer("ReLU"))
mnist_model.add_layer(LinearLayer(40,10,min_weight=-0.1,max_weight=0.1))

In [9]:

# epoch_count = 30
# alpha = 0.001
# train_data_size = x_train.shape[0]
# for epoch_number in range(epoch_count):
# 	print(f'Epoka: {epoch_number}')
# 	error = 0
# 	for i in range(train_data_size):
# 		if i % 1000 == 0:
# 			print(f'element zbioru treningowego {i}')
# 		y_array = np.zeros(10)
# 		y_array[int(y_train[i])] = 1
# 		error += mnist_model.fit(x_train[i, :], y_array, learning_rate=alpha)
# 	print(error / train_data_size)
#



In [10]:
#mnist_model.save_model("../saves/mnist_model1")
mnist_model.load_model("../saves/mnist_model1")

In [11]:

correctness , _ = mnist_model.evaluate(x_test,y_test)
print(f'poprawność : {correctness}')

poprawność : 0.856
